# 04 · Data analysis with a downloadable chart

Claude (Bedrock) analyzes a dataset by writing code in a sandbox, saves a chart
**file**, and the notebook downloads it with `files.read(format="bytes")` and
renders it inline — the artifact pattern for getting non-text output out of a sandbox.

**Concepts:** code as the medium (answers are computed, not imagined); artifacts
retrieved as bytes; the `agentcore-ci` backend ships numpy/pandas/matplotlib.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & tool
Bedrock client, an `execute_python` tool, and a prompt telling the model to save the figure to a known path.

In [ ]:
import os

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

CHART = "chart.png"   # workspace-relative so files.read() can fetch it
SYSTEM = [{"text": "You are a data analyst. Use execute_python (pandas + matplotlib, a "
           f"non-interactive backend) and save any figure to '{CHART}' with plt.savefig; "
           "do not call plt.show()."}]
TOOL_CONFIG = {"tools": [{"toolSpec": {
    "name": "execute_python",
    "description": "Run Python in a stateful sandbox kernel; returns stdout/stderr/errors.",
    "inputSchema": {"json": {"type": "object",
        "properties": {"code": {"type": "string"}}, "required": ["code"]}},
}}]}

### Tool + agent loop

In [ ]:
def run_tool(sbx, code):
    ex = sbx.run_code(code, on_stdout=lambda o: print("[py]", o, end=""))
    return f"ERROR {ex.error.name}: {ex.error.value}" if ex.error else ("".join(ex.logs.stdout) or "(ok)")


def agent(sbx, task):
    messages = [{"role": "user", "content": [{"text": task}]}]
    while True:
        resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM, messages=messages,
                                toolConfig=TOOL_CONFIG, inferenceConfig={"maxTokens": 4096, "temperature": 0})
        out = resp["output"]["message"]
        messages.append(out)
        if resp["stopReason"] != "tool_use":
            return out
        results = [{"toolResult": {"toolUseId": b["toolUse"]["toolUseId"],
                    "content": [{"text": run_tool(sbx, b["toolUse"]["input"]["code"])}]}}
                   for b in out["content"] if "toolUse" in b]
        messages.append({"role": "user", "content": results})

### Analyze a dataset, then download & show the chart

In [ ]:
from IPython.display import Image, display

SIGNUPS_CSV = """day,signups
Mon,120
Tue,145
Wed,98
Thu,175
Fri,210
Sat,60
Sun,55
"""

with Sandbox.create(region_name=REGION) as sbx:
    sbx.files.write("signups.csv", SIGNUPS_CSV)
    final = agent(sbx, "Load signups.csv, print total and average daily signups, and save a "
                       f"line chart of cumulative signups across the week to '{CHART}'.")
    for b in final["content"]:
        if "text" in b:
            print("\nClaude:", b["text"])
    png = sbx.files.read(CHART, format="bytes")

display(Image(data=bytes(png)))

### Recap & next
Scale this to scoring or datasets of tasks: `05-eval-harness`, `10-parallel-eval-runner`.